# Stack Mínimo de IA para Startups

Elige el modelo correcto, construye el backend mínimo y mide desde el día 1.

In [ ]:
import anthropic
import json
import time
import uuid

client = anthropic.Anthropic()
print("Stack mínimo listo")

## 1. Selector de modelo según volumen y presupuesto

In [ ]:
PRECIOS = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00, "label": "Haiku"},
    "claude-sonnet-4-6":         {"input": 3.00, "output": 15.00, "label": "Sonnet"},
    "claude-opus-4-7":           {"input": 15.00, "output": 75.00, "label": "Opus"}
}

def simular_costes_mensuales(
    llamadas_dia: int,
    tokens_input_promedio: int = 800,
    tokens_output_promedio: int = 400
) -> None:
    """Muestra tabla de costes mensuales por modelo."""
    print(f"\nCostes mensuales — {llamadas_dia} llamadas/día")
    print(f"({'~' + str(tokens_input_promedio) + ' tokens input / ~' + str(tokens_output_promedio) + ' output por llamada'})")
    print("-" * 55)
    print(f"{'Modelo':<10} {'Coste/llamada':>15} {'Coste mes':>12} {'Coste año':>12}")
    print("-" * 55)

    for modelo_id, p in PRECIOS.items():
        coste_llamada = (tokens_input_promedio * p['input'] + tokens_output_promedio * p['output']) / 1_000_000
        coste_mes = coste_llamada * llamadas_dia * 30
        coste_año = coste_mes * 12
        print(f"{p['label']:<10} ${coste_llamada:>13.5f} ${coste_mes:>10.2f} ${coste_año:>10.2f}")

# Diferentes etapas de crecimiento
for llamadas in [100, 1_000, 10_000, 100_000]:
    simular_costes_mensuales(llamadas)

## 2. Benchmark de calidad Haiku vs Sonnet en tu caso de uso

In [ ]:
def benchmark_modelos(system: str, casos: list[dict]) -> list[dict]:
    """Compara Haiku vs Sonnet en tus casos de uso reales."""
    modelos = [
        ("claude-haiku-4-5-20251001", "Haiku"),
        ("claude-sonnet-4-6", "Sonnet")
    ]
    resultados = []

    for caso in casos:
        fila = {"input": caso["input"][:60] + "..."}
        for modelo_id, label in modelos:
            inicio = time.perf_counter()
            resp = client.messages.create(
                model=modelo_id,
                max_tokens=300,
                system=system,
                messages=[{"role": "user", "content": caso["input"]}]
            )
            latencia = (time.perf_counter() - inicio) * 1000
            p = PRECIOS[modelo_id]
            coste = (resp.usage.input_tokens * p['input'] + resp.usage.output_tokens * p['output']) / 1_000_000

            fila[label] = {
                "output": resp.content[0].text[:120],
                "tokens": resp.usage.input_tokens + resp.usage.output_tokens,
                "latencia_ms": round(latencia),
                "coste_usd": round(coste, 5)
            }
        resultados.append(fila)
    return resultados


SYSTEM_DEMO = """Eres un asistente de análisis de contratos para PYMEs españolas.
Identifica cláusulas problemáticas e indica el nivel de riesgo: BAJO / MEDIO / ALTO.
Sé conciso y directo. Nunca des asesoramiento jurídico vinculante."""

CASOS_PRUEBA = [
    {"input": "Cláusula 5: El proveedor no será responsable de ningún daño directo o indirecto bajo ninguna circunstancia."},
    {"input": "Cláusula 8: Cualquier disputa se resolverá en tribunales de Nueva York bajo ley del estado de Delaware."},
    {"input": "Cláusula 2: El precio se actualizará anualmente según el IPC del INE."}
]

print("Benchmarking Haiku vs Sonnet...")
resultados = benchmark_modelos(SYSTEM_DEMO, CASOS_PRUEBA)

for r in resultados:
    print(f"\n📋 Input: {r['input']}")
    for label in ["Haiku", "Sonnet"]:
        d = r[label]
        print(f"  [{label}] ${d['coste_usd']:.5f} | {d['latencia_ms']}ms | {d['tokens']} tok")
        print(f"    → {d['output'][:100]}")

## 3. Backend mínimo con streaming

In [ ]:
from collections import defaultdict

# Rate limiter mínimo para el MVP
class RateLimiterSimple:
    def __init__(self, max_req: int = 20, ventana_seg: int = 60):
        self.max = max_req
        self.ventana = ventana_seg
        self._hist: dict = defaultdict(list)

    def ok(self, uid: str) -> bool:
        ahora = time.time()
        self._hist[uid] = [t for t in self._hist[uid] if ahora - t < self.ventana]
        if len(self._hist[uid]) >= self.max:
            return False
        self._hist[uid].append(ahora)
        return True


# Simulación del backend MVP
SYSTEM_MVP = """Eres un asistente de análisis de contratos para PYMEs españolas.
Identifica cláusulas problemáticas e indica el riesgo: BAJO / MEDIO / ALTO.
Sé conciso. Nunca des asesoramiento jurídico vinculante."""

SESIONES: dict = {}
limiter = RateLimiterSimple(max_req=5, ventana_seg=60)


def procesar_mensaje_mvp(user_id: str, mensaje: str, session_id: str = None) -> dict:
    """Simula el endpoint /api/chat del MVP."""

    if not limiter.ok(user_id):
        return {"error": "Rate limit excedido", "status": 429}

    session_id = session_id or str(uuid.uuid4())[:8]
    sesion = SESIONES.setdefault(session_id, {"msgs": [], "tokens": 0})
    sesion["msgs"].append({"role": "user", "content": mensaje})

    inicio = time.perf_counter()
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        system=SYSTEM_MVP,
        messages=sesion["msgs"]
    )
    latencia = (time.perf_counter() - inicio) * 1000

    texto = resp.content[0].text
    sesion["msgs"].append({"role": "assistant", "content": texto})
    sesion["tokens"] += resp.usage.input_tokens + resp.usage.output_tokens

    coste = sesion["tokens"] / 1_000_000 * 2.40

    print(json.dumps({
        "evento": "chat", "session": session_id, "user": user_id,
        "latencia_ms": round(latencia), "tokens_sesion": sesion["tokens"],
        "coste_sesion_usd": round(coste, 5)
    }))

    return {"session_id": session_id, "respuesta": texto, "latencia_ms": round(latencia)}


# Simular conversación multi-turno
print("Simulación de conversación MVP:")
print("-" * 50)

r1 = procesar_mensaje_mvp("user_ana", "Analiza: El proveedor no asume responsabilidad por lucro cesante.")
print(f"Respuesta: {r1['respuesta'][:150]}...\n")

r2 = procesar_mensaje_mvp("user_ana", "¿Qué alternativa propondrías?", session_id=r1["session_id"])
print(f"Respuesta: {r2['respuesta'][:150]}...")

## 4. Checklist de lanzamiento

In [ ]:
CHECKLIST_LANZAMIENTO = {
    "Producto": [
        ("System prompt versionado y testeado con 10+ casos", True),
        ("Rate limiting por usuario activo", True),
        ("Logging de cada llamada a la API", True),
        ("Límites de tokens por sesión configurados", True),
        ("Manejo de errores (API down, timeout)", False),
    ],
    "Negocio": [
        ("Precio definido antes de lanzar (aunque sea beta)", False),
        ("Stripe integrado desde el día 1", False),
        ("10 usuarios beta comprometidos", True),
        ("Mecanismo de feedback (thumbs up/down) activo", True),
        ("Alertas de coste configuradas", False),
    ],
    "Legal/Seguridad": [
        ("Términos de servicio publicados", False),
        ("Política de privacidad GDPR-compatible", False),
        ("Datos de usuarios no se envían a terceros sin consentimiento", True),
        ("Variables de entorno en .env (nunca en repositorio)", True),
    ]
}

print("CHECKLIST DE LANZAMIENTO MVP")
print("=" * 50)
total_ok = 0
total = 0

for categoria, items in CHECKLIST_LANZAMIENTO.items():
    ok_cat = sum(1 for _, v in items if v)
    total_ok += ok_cat
    total += len(items)
    print(f"\n{categoria} ({ok_cat}/{len(items)}):")
    for desc, hecho in items:
        icono = "✅" if hecho else "❌"
        print(f"  {icono} {desc}")

pct = round(total_ok / total * 100)
print(f"\n{'='*50}")
print(f"Puntuación: {total_ok}/{total} ({pct}%)")
if pct >= 80:
    print("✅ Listo para lanzar con primeros usuarios beta")
elif pct >= 60:
    print("⚠️  Completa los items críticos antes de lanzar")
else:
    print("❌ Necesitas más trabajo antes del lanzamiento")